# 11: Evaluation Metrics and the Bias-Variance Trade-off

**Author:** Dr Rob Lyon

**Version:** 1.0

## Code & License
Copyright © 2026 Robert Lyon. All Rights Reserved.

This notebook and all associated materials are the intellectual property of the author.

Permission is granted solely to read, study, and analyse this material for personal educational purposes. No other rights are granted.

Without the prior written consent of the author, you may not:

* Copy, reproduce, redistribute, publish, transmit, or display this work in whole or in part.
* Modify, adapt, transform, translate, or create derivative works based on this material.
* Incorporate any portion of this work into another project, publication, product, model, dataset, or codebase.
* Use this material for commercial purposes.
* Remove or alter this copyright notice.

All intellectual property rights, including copyright and any derivative rights, remain exclusively vested in the author.

Access to this material does not constitute a transfer of ownership, license, or any other intellectual property rights except as expressly stated above.

---

**Note:**

This notebook is designed to be viewed from within JupyterLab. The Table of Contents and "Back to Table of Contents" links rely on JupyterLab's anchor handling to jump between sections.

If you open this notebook in another tool, such as PyCharm or Visual Studio Code, these anchor links may not work as expected. You can still navigate the notebook in those tools by using the headings directly, most editors provide an outline or contents panel that lists them for you.

---

## Table of Contents

1. [Introduction](#1.-Introduction)
2. [Useful Links](#2.-Useful-Links)
3. [Libraries and Environment Setup](#3.-Libraries-and-Environment-Setup)
4. [Why Accuracy Alone Is Not Enough — A Recap](#4.-Why-Accuracy-Alone-Is-Not-Enough-—-A-Recap)
   - 4.1 [Training a Classifier to Ground the Metrics](#4.1-Training-a-Classifier-to-Ground-the-Metrics)
5. [Recall (Sensitivity / True Positive Rate)](#5.-Recall-(Sensitivity-/-True-Positive-Rate))
   - 5.1 [When recall matters most](#5.1-When-recall-matters-most)
   - 5.2 [Recall, precision, and the decision threshold](#5.2-Recall,-precision,-and-the-decision-threshold)
6. [Precision (Positive Predictive Value)](#6.-Precision-(Positive-Predictive-Value))
   - 6.1 [When precision matters most](#6.1-When-precision-matters-most)
   - 6.2 [The fundamental tension between precision and recall](#6.2-The-fundamental-tension-between-precision-and-recall)
7. [F1-Score: The Harmonic Mean of Precision and Recall](#7.-F1-Score:-The-Harmonic-Mean-of-Precision-and-Recall)
   - 7.1 [When to use F1](#7.1-When-to-use-F1)
   - 7.2 [The $F_\beta$ generalisation](#7.2-The-$F_\beta$-generalisation)
8. [Specificity, FPR, and G-Mean](#8.-Specificity,-FPR,-and-G-Mean)
   - 8.1 [Specificity (True Negative Rate)](#8.1-Specificity-(True-Negative-Rate))
   - 8.2 [False Positive Rate](#8.2-False-Positive-Rate)
   - 8.3 [G-Mean](#8.3-G-Mean)
9. [Metrics Designed for Imbalanced Data](#9.-Metrics-Designed-for-Imbalanced-Data)
   - 9.1 [Balanced Accuracy](#9.1-Balanced-Accuracy)
   - 9.2 [Cohen's Kappa](#9.2-Cohen's-Kappa)
   - 9.3 [Matthews Correlation Coefficient (MCC)](#9.3-Matthews-Correlation-Coefficient-(MCC))
10. [Putting It All Together: A Binary Classification Evaluation Workflow](#10.-Putting-It-All-Together:-A-Binary-Classification-Evaluation-Workflow)
    - 10.1 [Building the dataset](#10.1-Building-the-dataset)
    - 10.2 [Training the classifier](#10.2-Training-the-classifier)
    - 10.3 [Computing and reporting the metrics](#10.3-Computing-and-reporting-the-metrics)
11. [Multi-Class Metrics: Macro, Micro, and Weighted Averaging](#11.-Multi-Class-Metrics:-Macro,-Micro,-and-Weighted-Averaging)
    - 11.1 [Putting It All Together: A Multi-class Classification Evaluation Workflow](#11.1-Putting-It-All-Together:-A-Multi-class-Classification-Evaluation-Workflow)
12. [Bias and Variance: Two Sources of Error](#12.-Bias-and-Variance:-Two-Sources-of-Error)
13. [Diagnosing Underfitting and Overfitting](#13.-Diagnosing-Underfitting-and-Overfitting)
    - 13.1 [Diagnosing underfitting (high bias)](#13.1-Diagnosing-underfitting-(high-bias))
    - 13.2 [Diagnosing overfitting (high variance)](#13.2-Diagnosing-overfitting-(high-variance))
    - 13.3 [Reading the generalisation gap](#13.3-Reading-the-generalisation-gap)
14. [Regularisation: Controlling Complexity](#14.-Regularisation:-Controlling-Complexity)
    - 14.1 [L2 (Ridge) Regularisation](#14.1-L2-(Ridge)-Regularisation)
    - 14.2 [L1 (Lasso) Regularisation](#14.2-L1-(Lasso)-Regularisation)
    - 14.3 [Applying Regularisation in scikit-learn: A Step-by-Step Workflow](#14.3-Applying-Regularisation-in-scikit-learn:-A-Step-by-Step-Workflow)
    - 14.4 [Regularisation Across Different Classifiers](#14.4-Regularisation-Across-Different-Classifiers)
      - 14.4.1 [Classifiers with explicit L1/L2 regularisation](#14.4.1-Classifiers-with-explicit-L1/L2-regularisation)
      - 14.4.2 [Classifiers with structural regularisation](#14.4.2-Classifiers-with-structural-regularisation)
      - 14.4.3 [Naive Bayes: a special case](#14.4.3-Naive-Bayes:-a-special-case)
      - 14.4.4 [The common principle](#14.4.4-The-common-principle)
15. [Summary](#15.-Summary)
16. [References](#16.-References)

---

## 1. Introduction

This notebook has two major themes. The first is **evaluation metrics** that go beyond basic measures of accuracy: recall, precision, F1-score, specificity, the false positive rate (FPR), G-mean, and three imbalance-aware measures, balanced accuracy, Matthews Correlation Coefficient (MCC), and Cohen's Kappa. We work through each metric in turn, looking at when it matters and when it can mislead. The second theme is **why errors happen**: the bias-variance trade-off, underfitting, overfitting, and regularisation as the principal tool for managing that trade-off. These two themes are closely connected. The metrics tell us *what kind* of errors a model is making, and the bias-variance framework tells us *why* they happen and *how* to address them.

### Learning Objectives

By the end of this notebook you should be able to:

- Explain why accuracy is an insufficient metric for evaluating classifiers on imbalanced datasets.
- Define and compute recall, precision, specificity, FPR, F1-score, G-mean, balanced accuracy, MCC, and Cohen's Kappa from a confusion matrix.
- Describe the precision-recall trade-off and explain how the decision threshold affects it.
- Apply macro, micro, and weighted averaging to extend binary metrics to multi-class problems.
- Explain the bias-variance decomposition of prediction error and describe the symptoms of underfitting and overfitting.
- Use regularisation (L1 and L2) to control model complexity and reduce overfitting.
- Select appropriate regularisation strength using cross-validation.


---

## 2. Useful Links

| Link | Description |
|------|-------------|
| [scikit-learn: Model Evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html) | Comprehensive reference for all evaluation metrics in scikit-learn, including precision, recall, F1, MCC, and Cohen's Kappa. |
| [scikit-learn: Ridge and Lasso](https://scikit-learn.org/stable/modules/linear_model.html#ridge-regression-and-classification) | Documentation for Ridge and Lasso regularisation, including mathematical formulation and practical guidance. |
| [scikit-learn: LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) | API reference for Logistic Regression, covering regularisation type (`penalty`) and strength (`C`). |
| [Understanding the Bias-Variance Tradeoff](https://scott.fortmann-roe.com/docs/BiasVariance.html) | An accessible and widely cited visual introduction to the bias-variance trade-off with interactive demonstrations. |
| [Wikipedia: Matthews Correlation Coefficient](https://en.wikipedia.org/wiki/Phi_coefficient) | Background on MCC (also known as the Phi coefficient), its mathematical properties, and why it is considered a reliable single-number binary metric. |



---

## 3. Libraries and Environment Setup

### 3.1 Working Environment

To run the code in this notebook, you will need a Python environment with the required libraries listed below installed.

The easiest way to get started is to use the **Project IO** Docker container, which provides a fully configured and tested environment containing all required Python packages and supporting dependencies. Instructions for downloading and running the container can be found in the **project-io** GitHub repository:

https://github.com/scienceguyrob/project-io

Using the Docker container ensures that the notebooks run in exactly the same environment in which they were developed and tested.

If you prefer not to use Docker, a good alternative is to install the [Anaconda Distribution](https://www.anaconda.com/products/distribution). You can then create a new Python environment and install the required libraries using either `conda install` or `pip install`.

### 3.2 Libraries Used in This Notebook

| Library | Purpose | Documentation |
|---------|---------|---------------|
| **NumPy** (`numpy`) | Fast numerical arrays, mathematical functions, random number generation. | [numpy.org](https://numpy.org/doc/stable/) |
| **Pandas** (`pandas`) | Tabular data structures (`DataFrame`) for organising metric comparisons. | [pandas.pydata.org](https://pandas.pydata.org/docs/) |
| **Matplotlib** (`matplotlib.pyplot`) | Standard Python plotting library. Used for all figures and visualisations. | [matplotlib.org](https://matplotlib.org/stable/) |
| **Seaborn** (`seaborn`) | Statistical visualisation built on Matplotlib. Used primarily for confusion matrix heatmaps. | [seaborn.pydata.org](https://seaborn.pydata.org/) |
| **scikit-learn** (`sklearn`) | Primary machine learning library: classifiers, metrics, preprocessing, dataset generators. | [scikit-learn.org](https://scikit-learn.org/stable/) |
| **ipywidgets** (`ipywidgets`) | Adds interactive UI controls (sliders, dropdowns, etc.) to Jupyter notebooks. We use it to create sliders for adjusting plot parameters in real time. | [ipywidgets.readthedocs.io](https://ipywidgets.readthedocs.io/en/stable/) |
| **ipympl** (`matplotlib widget` backend) | Enables interactive, live-updating Matplotlib figures inside Jupyter. Required for smooth slider updates without redrawing the entire plot. | [matplotlib.org/ipympl](https://matplotlib.org/ipympl/) |

### 3.3 Imports

All library imports for this notebook are placed in the cell below. This is a deliberate best practice: keeping all imports in one place makes it easy to see what is required and avoids confusing errors when an import cell is skipped.

> ⚠️ **You must run the cell below before executing any other code cell in this notebook.** If you skip this cell, all subsequent code will raise a `NameError`.

In [ ]:
# Listing 1.
# ─────────────────────────────────────────────────────────────────────────────
# All library imports for this notebook are placed here for clarity.
# Run this cell first before executing any code.
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np                                   # numerical arrays and mathematical functions
import pandas as pd                                  # tabular data structures for metric comparisons
import matplotlib.pyplot as plt                      # plotting and visualisation
import matplotlib.patches as mpatches               # custom legend handles for patch-based plots
import seaborn as sns                                # statistical visualisation (heatmaps, etc.)

import matplotlib
matplotlib.rcParams['figure.max_open_warning'] = 50

# scikit-learn: datasets
from sklearn.datasets import make_classification, load_breast_cancer, load_iris

# scikit-learn: classifiers
from sklearn.tree import DecisionTreeClassifier      # decision tree for bias-variance demonstrations
from sklearn.neighbors import KNeighborsClassifier   # k-nearest neighbours
from sklearn.linear_model import (
    LogisticRegression,                              # logistic regression with L1/L2 regularisation
    Ridge,                                           # ridge (L2) regression
    Lasso,                                           # lasso (L1) regression
)
from sklearn.svm import SVC                          # support vector classifier
from sklearn.ensemble import RandomForestClassifier  # ensemble method
from sklearn.naive_bayes import GaussianNB           # Gaussian Naive Bayes

# scikit-learn: preprocessing and pipelines
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline           # chains preprocessing and model into one object

# scikit-learn: model selection
from sklearn.model_selection import (
    train_test_split,                                # split data into training and test sets
    StratifiedKFold,                                 # k-fold CV preserving class proportions
    cross_val_score,                                 # evaluate a model using cross-validation
)

# scikit-learn: evaluation metrics
from sklearn.metrics import (
    confusion_matrix,                                # compute the N x N confusion matrix
    ConfusionMatrixDisplay,
    accuracy_score,                                  # overall fraction of correct predictions
    precision_score,                                 # TP / (TP + FP)
    recall_score,                                    # TP / (TP + FN)
    f1_score,                                        # harmonic mean of precision and recall
    matthews_corrcoef,                               # Matthews Correlation Coefficient (MCC)
    balanced_accuracy_score,                         # mean of per-class recall values
    cohen_kappa_score,                               # agreement above chance
)

rng = np.random.default_rng(42)                     # reproducible random number generator

# Confirm successful import
print('Libraries loaded successfully.')



---

## 4. Why Accuracy Alone Is Not Enough — A Recap

🔙 [Back to Table of Contents](#table-of-contents)

In Notebook 10 we reviewed the confusion matrix. We learned that a model which always predicts the majority class can achieve very high accuracy on an imbalanced dataset while being completely useless in practice. The confusion matrix shows us *where* the errors are, but we need **numerical metrics** to quantify and compare them systematically. You can see the basic confusion matrix below.

$$
\begin{array}{ |c|c|c|}
\hline
 & \textbf{Predicted Positive} & \textbf{Predicted Negative} \\ \hline
\textbf{Actual Positive} & TP & FN \\ \hline
\textbf{Actual Negative} & FP & TN \\ \hline
\end{array}
$$

This is the confusion matrix for a binary (two-class) classifier that sorts every prediction into one of four categories:

- **True Positives (TP):** Correctly predicted positive instances.
- **False Negatives (FN):** Actual positives incorrectly predicted as negative.
- **False Positives (FP):** Actual negatives incorrectly predicted as positive.
- **True Negatives (TN):** Correctly predicted negative instances.

Accuracy is simply given by the proportion of all predictions that were correct:

$$Accuracy = \frac{TP + TN}{TP + TN + FP + FN}$$

Before going further, it is worth pausing on what "positive" and "negative" actually mean here, because the everyday meanings of those words can be misleading. In binary classification, the **positive class** is simply the class we have chosen to call the **target**, the thing we are trying to detect or identify, and the **negative class** is everything else, the **non-target** class. Here "positive" does not mean "good" and "negative" does not mean "bad": if the target is to detect a rare disease, a positive result means the disease was detected. It is simply the outcome the model is being asked to find. Whichever class is designated as the target is a choice made when framing the problem, not a property of the data itself.

A useful way to think about it is that the positive class is the "thing we are looking for", and the negative class is "everything else". In an imbalanced dataset, the target (positive) class is very often also the minority class, the rarer of the two (e.g. fraud, disease, equipment faults, and so on). This is why the terms "positive class" and "minority class" are frequently used together, but they are not the same thing by definition, only by common practice.

With that framing in mind, Figure 1 below gives you a hands-on way to get a feel for why accuracy on its own can be so misleading. It presents the same confusion matrix layout shared in Notebook 10. It lets you build a confusion matrix using four sliders:

| Slider | Controls | What it means |
|---|---|---|
| `n (samples)` | the total dataset size | ranges from 1 to 1,000,000 samples, on a log scale, so you can see whether the patterns below hold at small and large scale |
| `% positive` | the class balance | the proportion of the dataset that belongs to the target (positive) class, the source of the imbalance discussed above |
| `Recall (TP rate)` | how many actual positives are caught | the proportion of actual target (positive) cases the model correctly flags as positive, $\frac{TP}{TP+FN}$ |
| `FP rate` | how many actual negatives are wrongly flagged | the proportion of actual non-target (negative) cases the model incorrectly flags as positive, $\frac{FP}{FP+TN}$ |

Recall and FP rate are two of the metrics this notebook builds towards properly later on, so do not worry about memorising their definitions yet, for now just treat them as two dials that let you control how the model behaves towards the target and non-target classes respectively.

As you move these sliders, watch the confusion matrix update on the left, and the accuracy, precision, and recall figures update on the right. In particular, try the following:

- Set `% positive` low (say 5%) and both `Recall (TP rate)` and `FP rate` to 0%. This reconstructs the "always predict the majority class" model from Notebook 10: notice that accuracy can still be very high, while recall is 0% and precision is undefined, because the model never predicts the target (positive) class at all.
- Now raise `Recall (TP rate)` while keeping `% positive` low. Notice that accuracy barely changes, even though the model is now correctly identifying target cases it previously missed entirely.

> 📌 **What to notice:** with an imbalanced dataset, accuracy can stay almost flat across very different models, even ones that go from completely ignoring the target class to catching most of it. This is the core problem with relying on accuracy alone, and it is exactly what the metrics introduced later in this notebook are designed to fix.

In [ ]:
# Listing 2.
%matplotlib widget
from visualisations.Figure_1 import show
show()

---

### 4.1 Training a Classifier to Ground the Metrics

Metrics only make sense when they are attached to a real set of predictions, so before working through precision, recall, and the rest, we need a classifier to evaluate. Rather than using a real-world dataset that would require cleaning and domain knowledge to interpret, we'll use a **synthetic binary classification dataset** generated by scikit-learn. This keeps the focus on the metrics themselves rather than on the data.

The classifier we'll use for now is a logistic regression model, which you will recognise from Notebook 5. It outputs a probability score for each test instance, which we then threshold at 0.5 to produce a hard predicted label. Once we have those predictions, we pass them through scikit-learn's `confusion_matrix` function to extract the four counts, $TP$, $TN$, $FP$, and $FN$, that every metric in this notebook is built from.

The dataset is constructed with a 10% positive class, meaning only one in ten samples belongs to the target class. This is a deliberate choice: it recreates the kind of class imbalance discussed above, and it means the metrics we compute here will tell a more interesting story than they would on a balanced dataset.

> 📓 **Notebook 5 recap:** logistic regression models the probability that an instance belongs to the positive class. Given a feature vector $\mathbf{x}$, it outputs $\hat{p} = \sigma(\mathbf{w}^\top \mathbf{x} + b)$, where $\sigma$ is the sigmoid function that squashes any real value into the range $(0, 1)$. The predicted class is then determined by comparing $\hat{p}$ against a decision threshold, typically 0.5 by default.

Run the cell below. It builds the dataset, trains and evaluates the classifier, and stores the values `TP`, `TN`, `FP`, and `FN` that will be used in every section that follows. The printed output gives you a first look at the raw counts before we turn them into metrics.

In [ ]:
# Listing 3.
# ── Build dataset ─────────────────────────────────────────────────────────────
# A synthetic binary classification problem with a 10% positive class,
# matching the imbalance scenario discussed in Section 4. Two informative
# features per class and a small amount of label noise (flip_y) keep the
# problem realistic: the classifier will be good but not perfect, which
# makes the metrics more interesting to inspect.
POSITIVE_FRACTION = 0.10   # 10% positive (minority) class
N_SAMPLES         = 2_000  # total dataset size
RANDOM_STATE      = 42     # fixed for reproducibility across notebook runs

X, y = make_classification(
    n_samples=N_SAMPLES,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    n_clusters_per_class=2,
    weights=[1 - POSITIVE_FRACTION, POSITIVE_FRACTION],
    flip_y=0.05,           # 5% label noise: avoids a perfectly separable problem
    random_state=RANDOM_STATE,
)

# ── Train / test split ────────────────────────────────────────────────────────
# stratify=y preserves the 10/90 class ratio in both splits, which is
# essential with imbalanced data: without it, random chance could put very
# few positive cases in the test set and make metrics unreliable.
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE,
)

# ── Standardise features ──────────────────────────────────────────────────────
# Fit the scaler on the training set only, then apply the same transformation
# to the test set. Fitting on the full dataset before splitting would leak
# information about the test set into training, a subtle but important error
# covered in Notebook 10.
scaler  = StandardScaler()
X_tr_s  = scaler.fit_transform(X_tr)
X_te_s  = scaler.transform(X_te)

# ── Train classifier ──────────────────────────────────────────────────────────
# Logistic regression with a generous iteration limit: the default of 100
# iterations can fail to converge on standardised data with multiple
# features, producing a ConvergenceWarning and a poorly fitted model.
clf = LogisticRegression(max_iter=1_000, random_state=RANDOM_STATE)
clf.fit(X_tr_s, y_tr)
y_pred = clf.predict(X_te_s)

# ── Extract confusion matrix counts ───────────────────────────────────────────
# confusion_matrix returns a 2x2 array laid out as:
#
#        Predicted Neg   Predicted Pos
#   Actual Neg    TN          FP        (row 0)
#   Actual Pos    FN          TP        (row 1)
#
# Indexing [row, col] = [actual class, predicted class], so:
#   TN = [0,0], FP = [0,1], FN = [1,0], TP = [1,1]
#
# TP, TN, FP, FN are used throughout the rest of this notebook.
cm          = confusion_matrix(y_te, y_pred)
TN, FP, FN, TP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]

print('Test set summary')
print(f'  Total test samples : {len(y_te):,}')
print(f'  Actual positives   : {y_te.sum():,}  ({y_te.mean():.1%} of test set)')
print(f'  Actual negatives   : {(y_te == 0).sum():,}')
print()
print('Confusion matrix counts')
print(f'  TP = {TP}   FN = {FN}')
print(f'  FP = {FP}   TN = {TN}')
print()
print('Quick sanity check against sklearn')
print(f'  recall_score    : {recall_score(y_te, y_pred):.4f}')
print(f'  precision_score : {precision_score(y_te, y_pred):.4f}')

---

The cell above builds the confusion matrix and unpacks it into four named variables. It is worth understanding exactly what those two lines do, because you will write this pattern in almost every evaluation workflow you build:

```python
cm             = confusion_matrix(y_te, y_pred)
TN, FP, FN, TP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
```

Here `confusion_matrix(y_te, y_pred)` takes the true labels and the predicted labels and returns a 2×2 NumPy array. The rows correspond to the **actual** class and the columns correspond to the **predicted** class, following the standard orientation used throughout this notebook. For a binary problem with classes 0 (negative) and 1 (positive), the array is laid out like this:

$$\begin{array}{r|cc} & \textbf{Predicted 0} & \textbf{Predicted 1} \\ \hline \textbf{Actual 0} & TN & FP \\ \textbf{Actual 1} & FN & TP \end{array}$$

In NumPy index notation, where `cm[row, col]` means row = actual class and col = predicted class:

| Index | Meaning | Why |
|---|---|---|
| `cm[0, 0]` | $TN$ | actual 0, predicted 0: correctly labelled negative |
| `cm[0, 1]` | $FP$ | actual 0, predicted 1: a negative wrongly called positive |
| `cm[1, 0]` | $FN$ | actual 1, predicted 0: a positive that was missed |
| `cm[1, 1]` | $TP$ | actual 1, predicted 1: correctly labelled positive |

The second line simply unpacks those four positions into named variables in one assignment, which is more readable than scattering `cm[1,1]` and `cm[0,1]` throughout the code. From this point on, every metric in the notebook is expressed directly in terms of `TP`, `TN`, `FP`, and `FN`, so it is always clear which part of the confusion matrix is being used and why.

> ⚠️ **Warning:** scikit-learn's `confusion_matrix` orders the rows and columns by the **sorted unique values** of the label array. For standard binary labels of 0 and 1 this always produces the layout above, with the negative class (0) in row 0 and the positive class (1) in row 1. If your labels are strings, or integers that do not start at 0, the ordering may differ and the index assignments above will be wrong. You can make the ordering explicit by passing `labels=[0, 1]` (or your chosen class order) as a keyword argument, which is good practice whenever there is any ambiguity.

---

## 5. Recall (Sensitivity / True Positive Rate)

🔙 [Back to Table of Contents](#table-of-contents)

The first question we need to ask about a classifier's behaviour on the positive (target) class is straightforward: **of all the actual positives in the dataset, what fraction did we correctly identify?** This is what recall measures.

$$\text{Recall} = \frac{TP}{TP + FN}$$

The denominator $TP + FN$ is the total number of actual positives, the full set of target cases in the data. Those cases end up in exactly one of two places: either we caught them ($TP$, true positives) or we missed them ($FN$, false negatives). Recall is simply the proportion we caught. A recall of 1.0 means every positive case was found; a recall of 0.0 means none were.

Recall is also widely known as **sensitivity** and as the **true positive rate (TPR)**. All three names refer to the same quantity. The different names reflect the different communities that use it: clinicians tend to say sensitivity, machine learning practitioners tend to say recall, and the signal-detection literature tends to say true positive rate. You will encounter all three, so it is worth knowing they are interchangeable.

### 5.1 When recall matters most

Recall becomes the priority metric in any application where **missing a positive case is the most costly kind of error**. Consider what a false negative actually means in practice:

| Domain | A false negative is... | The cost |
|---|---|---|
| Medical screening | a missed diagnosis | delayed or absent treatment, potentially fatal |
| Fraud detection | an undetected fraudulent transaction | direct financial loss |
| Fault detection | an undetected equipment failure | safety risk or unplanned downtime |
| Content moderation | harmful content that was not removed | exposure and reputational harm |

In all of these cases, the cost of a miss is much higher than the cost of a false alarm. A recall-focused model will therefore be tuned to catch as many true positives as possible, even if that means raising more false alarms (lower precision) in the process. This trade-off between recall and precision is one of the central tensions in classifier evaluation, and it comes up in virtually every real-world deployment decision.

### 5.2 Recall, precision, and the decision threshold

Most classifiers do not output a hard class label directly. Instead, they output a **probability score**, a number between 0 and 1 representing the model's estimated probability that the instance belongs to the positive class. The predicted label is then obtained by comparing that score against a **decision threshold**: if the score is at or above the threshold, the instance is labelled positive; otherwise it is labelled negative.

The default threshold is almost always 0.5, but there is nothing special about that value. Lowering the threshold means more instances are labelled positive, which tends to raise recall (fewer true positives are missed) but lower precision (more false alarms creep in). Raising the threshold does the opposite: fewer instances are labelled positive, so the classifier is more selective, precision tends to rise, but recall tends to fall as some true positives drop below the threshold and become false negatives.

This is the **precision–recall trade-off**, and it is a direct consequence of the threshold choice rather than a fixed property of the model. Understanding it is essential before comparing models on either metric in isolation.

The code below computes recall manually from the confusion matrix counts extracted in Section 4.1 and confirms the result against scikit-learn. The figure reuses the same fitted classifier and test set to plot how recall and precision vary as the decision threshold is swept from near 0 to near 1.

> 📌 **What to notice:** in the left panel, recall falls as the threshold rises, the direct consequence of the classifier becoming more conservative about labelling instances as positive. The dot marks where the default threshold of 0.5 sits on that curve. In the right panel, watch how the two lines move in roughly opposite directions: as the threshold rises, recall falls while precision tends to rise. There is no threshold at which both are simultaneously maximised, and choosing where to operate on that curve is a decision this notebook returns to in the sections on the PR curve.

In [ ]:
# Listing 4.
%matplotlib widget
from visualisations.Figure_2 import show
show(clf, X_te_s, y_te) # These parameters where created in the previous code cell. Be sure to run that first!

---

## 6. Precision (Positive Predictive Value)

🔙 [Back to Table of Contents](#table-of-contents)

The complementary question to recall is this: **of everything we labelled as positive, what fraction actually was positive?**

$$\text{Precision} = \frac{TP}{TP + FP}$$

Where recall's denominator is the total number of actual positives ($TP + FN$, everything in the target class), precision's denominator is the total number of positive *predictions* ($TP + FP$, everything the model chose to label positive). Of those predictions, some were correct ($TP$) and some were false alarms ($FP$). Precision is simply the proportion that were correct. A precision of 1.0 means every positive prediction was justified; a precision of 0.0 means none of them were.

### 6.1 When precision matters most

Precision becomes the priority metric in any application where **raising a false alarm is the most costly kind of error**. Consider what a false positive actually means in practice:

| Domain | A false positive is... | The cost |
|---|---|---|
| Spam filtering | a legitimate email blocked | the user misses important communications |
| Drug approval | an unsafe drug cleared for use | patients exposed to harm |
| Legal proceedings | an innocent person convicted | a fundamental injustice |
| Surgical intervention | an unnecessary procedure performed | patient risk with no benefit |

In all of these cases, acting on a wrong positive prediction causes direct harm, whereas missing a positive (a false negative) is the lesser evil. A precision-focused model will therefore be tuned to only predict positive when it is confident, even if that means leaving some true positives unlabelled.

### 6.2 The fundamental tension between precision and recall

Precision and recall pull in opposite directions. It is a direct consequence of where each metric's denominator comes from.

You can always force recall to 1.0 by predicting positive for every single sample, because you will then catch every true positive by definition. But precision will collapse, since your predicted positives now include every false positive in the dataset too. Conversely, you can achieve precision of 1.0 by only predicting positive on the single instance you are most certain about, but recall will be near zero because almost every other true positive goes unlabelled.

Figure 3 below makes this tension concrete in two ways. The left panel shows a Venn diagram of the two denominators side by side, using the actual counts from the classifier trained in Section 4.1, so you can see exactly which predictions each metric is measuring over. The right panel plots three operating points in precision-recall space to illustrate that the balance between the two is a choice, not a fixed property of the model.

In [ ]:
# Listing 5.
%matplotlib widget
from visualisations.Figure_3 import show
show(TP, FP, FN, TN) # Remember, these values are computed earlier on.


---

## 7. F1-Score: The Harmonic Mean of Precision and Recall

🔙 [Back to Table of Contents](#table-of-contents)

Precision and recall each measure something important, but neither is sufficient on its own. A model that predicts positive for every sample achieves perfect recall while having worthless precision. A model that predicts positive for only its single most confident case achieves perfect precision while missing almost everything. What we really want is a single number that is only high when *both* are high simultaneously.

The obvious candidate is the arithmetic mean of the two scores, but it fails in a telling way. Imagine a model evaluated on an imbalanced dataset that is so conservative, it only ever makes a single positive prediction, and gets it right. Its precision is 1.0 (every positive prediction was correct) but its recall is close to zero at 0.01 (it missed almost every actual positive in the dataset). The arithmetic mean of those two scores is $\frac{1.0 + 0.01}{2} = 0.505$, which sounds like a fair, middling result, but in practice this is a useless model. The arithmetic mean lets a perfect score on one metric hide a catastrophic score on the other.

The **harmonic mean** does not have this flaw. It is dominated by whichever value is smaller, so a large value on one metric cannot rescue a small value on the other. The F1-score is defined as the harmonic mean of precision and recall:

$$F_1 = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

Applying this to the same example:

$$F_1 = \frac{2 \times 1.0 \times 0.01}{1.0 + 0.01} \approx 0.02$$

This correctly reflects that the model is nearly useless. The same arithmetic works in reverse: a model with precision 0.01 and recall 1.0 also scores $F_1 \approx 0.02$. Only when both metrics are genuinely high does $F_1$ become high. $F_1 = 1.0$ requires both precision and recall to be perfect; $F_1 = 0$ if either is zero.

### 7.1 When to use F1

F1 is most appropriate when you are working with an imbalanced dataset and both kinds of error carry real costs: missing a positive (false negative) and incorrectly flagging a negative (false positive) are both genuinely bad outcomes. In that setting, F1 gives a single summary score that reflects how well the model handles the positive class without being inflated by its performance on the easy majority class.

F1 is not always the right choice. If false positives and false negatives carry very different costs, you may deliberately want to prioritise one over the other, in which case optimising for F1 will push you toward a balance you do not actually want. In those cases the individual precision and recall scores, or the $F_\beta$ score introduced below, are more informative.

### 7.2 The $F_\beta$ generalisation

$F_1$ treats precision and recall as equally important. The $F_\beta$ score relaxes this assumption by introducing a parameter $\beta$ that controls how much weight recall receives relative to precision:

$$F_\beta = \frac{(1 + \beta^2) \cdot \text{Precision} \cdot \text{Recall}}{\beta^2 \cdot \text{Precision} + \text{Recall}}$$

When $\beta = 1$ this reduces to the standard $F_1$ score. When $\beta > 1$, recall is weighted more heavily, useful in applications like medical screening where missing a positive is the more serious error. When $\beta < 1$, precision is weighted more heavily, appropriate when false alarms are the more costly outcome. The two most commonly used variants are $F_2$ (recall twice as important as precision) and $F_{0.5}$ (precision twice as important as recall).

---

Figure 4 below shows two views of the same idea. The table on the left compares the arithmetic mean and F1 score across five precision-recall combinations, including the actual classifier from Section 4.1. The pattern is consistent: wherever one metric is very high and the other very low, the arithmetic mean stays deceptively moderate while F1 collapses. The "High P, low R" row is the starkest example, a model catching only 1% of actual positives scores an arithmetic mean of 0.500 but an F1 of just 0.020.

Our classifier sits at the bottom of the table with F1 = 0.076. This is a direct consequence of the class imbalance: at the default threshold of 0.5 the model is conservative enough about the minority class that its recall is only 0.042, and F1 correctly penalises that near-total failure to find actual positives, pulling the combined score down to near zero despite a precision of 0.429.

The heatmap on the right makes this geometry visible across the full precision-recall space. The red regions in the bottom-right and top-left corners are exactly the high-P-low-R and low-P-high-R traps the table illustrates. Only the top-right corner, where both metrics are simultaneously high, produces a strong F1 score, which is precisely the behaviour the harmonic mean was designed to enforce.

In [ ]:
# Listing 6.
# Compute the metrics directly.
precision = TP / (TP + FP)
recall = TP / (TP + FN)

%matplotlib widget
from visualisations.Figure_4 import show
show(precision, recall)



---

## 8. Specificity, FPR, and G-Mean

🔙 [Back to Table of Contents](#table-of-contents)

The metrics covered so far, recall, precision, and F1, all focus exclusively on the positive (minority) class. They measure how well the classifier finds actual positives and how reliable its positive predictions are, but they say nothing about how the classifier behaves on the negative (majority) class. This section introduces three quantities that bring the negative class back into the picture.

### 8.1 Specificity (True Negative Rate)

Where recall asks "of all the actual positives, how many did we catch?", specificity asks the equivalent question for the negative class: **of all the actual negatives, how many did we correctly leave alone?**

$$\text{Specificity} = \frac{TN}{TN + FP}$$

The denominator $TN + FP$ is the total number of actual negatives. Some of those were correctly labelled negative ($TN$, true negatives) and some were incorrectly flagged as positive ($FP$, false positives). Specificity is the proportion that were correctly handled. A specificity of 1.0 means the classifier raised no false alarms at all; a specificity of 0.0 means every actual negative was wrongly flagged as positive.

Specificity is also called the **true negative rate (TNR)**. Just as recall and sensitivity are two names for the same quantity, TNR and specificity are interchangeable.

### 8.2 False Positive Rate

The false positive rate is the direct complement of specificity: the fraction of actual negatives that were **incorrectly** labelled positive.

$$\text{FPR} = \frac{FP}{FP + TN} = 1 - \text{Specificity}$$

Because FPR and specificity always sum to exactly 1.0, they carry the same information from opposite directions. Specificity measures what the classifier gets right on negatives; FPR measures what it gets wrong. A FPR of 0 means no false alarms; a FPR of 1.0 means every actual negative was incorrectly flagged.

The FPR matters particularly in the context of the ROC curve (covered in Notebook 12), where it forms the x-axis. Plotting TPR against FPR as the decision threshold varies traces out a curve that summarises the full range of trade-offs the classifier can operate at, which is why FPR appears prominently in Figure 5 above.

> ⚠️ **Warning:** FPR and false discovery rate (FDR) are easily confused but measure very different things. FPR is $FP / (FP + TN)$, calculated over actual negatives. FDR is $FP / (FP + TP)$, calculated over predicted positives, and is the complement of precision. Mixing them up is a common mistake when reading papers that use both.

### 8.3 G-Mean

F1 balances precision and recall, but both of those metrics are calculated entirely over the positive class. A classifier could achieve a decent F1 score while performing very poorly on the negative class, and F1 would not penalise it at all. G-Mean addresses this by explicitly requiring good performance on **both** classes simultaneously.

$$\text{G-Mean} = \sqrt{\text{Sensitivity} \times \text{Specificity}} = \sqrt{TPR \times TNR}$$

The geometric mean has the same property as the harmonic mean in F1: it is dominated by whichever of its two inputs is smaller. A classifier that ignores the minority class entirely has $TPR = 0$, giving $\text{G-Mean} = \sqrt{0 \times 1} = 0$, regardless of how high its specificity is. The same is true in reverse: a classifier that flags everything as positive has $TNR = 0$, giving $\text{G-Mean} = 0$ regardless of its recall. Both classes must be handled well for G-Mean to be high.

G-Mean is particularly useful on imbalanced datasets where the negative class is large enough that even a poor classifier can achieve reasonable specificity simply by getting most negatives right. In that setting, F1 alone may be misleading, and G-Mean provides a complementary check that the positive class is not being neglected in order to inflate performance on the easy majority.

---

Figure 5 below makes the relationship between these three rates concrete using the classifier from Section 4.1. The left panel plots recall (TPR), specificity (TNR), and FPR against the decision threshold simultaneously. The mirror-image relationship between TNR and FPR is immediately visible: the two lines move in opposite directions by exactly the same amount at every threshold, and at any point on the x-axis they sum to 1.0. Also notice that TPR and FPR tend to move in the same direction as the threshold falls: lowering the threshold catches more true positives, but it also flags more false positives, and the left panel shows both effects at once.

The right panel takes those same (FPR, TPR) pairs and redraws them with FPR on the x-axis and TPR on the y-axis, collapsing the threshold into a single curve. This is the ROC curve, which will be covered in detail in Notebook 12. It is introduced here because understanding it requires being comfortable with FPR and TPR as a pair, and Figure 5 is the clearest way to see where the curve comes from. A random classifier that ignores the features entirely and assigns labels by chance produces the diagonal dashed line; any useful classifier should sit above it. The upper-left corner, where TPR is high and FPR is low simultaneously, is the ideal operating region.

In [ ]:
# Listing 7.
%matplotlib widget
from visualisations.Figure_5 import show
show(clf, X_te_s, y_te)

---

The code below computes G-Mean for the classifier from Section 4.1 and then compares it against four hypothetical operating points, ranging from a classifier that ignores the minority class entirely to one that ignores the majority class. Running through the table by eye before looking at the numbers is a useful exercise: for each row, ask yourself whether the sensitivity and specificity combination sounds like a useful model, and then check whether G-Mean agrees with your intuition.

The results are clear. Our model has a specificity of 0.992, meaning it correctly leaves 99.2% of actual negatives alone, which sounds impressive in isolation. But its sensitivity is only 0.042, and **G-Mean = 0.203**, correctly reflecting that the model is catching just 4% of actual positives is not performing well overall. Regardless of how clean it is on the negative class.

The "ignores minority class" and "ignores majority class" rows make the same point from opposite directions. Both are poor classifiers, and G-Mean penalises both: 0.222 and 0.308 respectively. Neither can hide its weakness behind its strength. The "balanced" row shows what a genuinely good classifier looks like in G-Mean terms: when both sensitivity and specificity are high, G-Mean rises to 0.825.

The bottom row is the limiting case. A classifier that always predicts negative achieves perfect specificity of 1.0 because it never raises a false alarm on a negative, but its sensitivity is 0.0 because it catches nothing at all. G-Mean collapses to 0.0, which is the correct verdict on a model that has completely abdicated its job of finding the positive class.

In [ ]:
# Listing 8.
# ── G-Mean: balancing recall across both classes ──────────────────────────────
#
# F1 only considers the positive (minority) class. G-Mean extends the idea
# of "both things must be high" to cover both classes simultaneously:
#
#   G-Mean = sqrt(Sensitivity * Specificity)
#          = sqrt(TPR * TNR)
#
# Like the harmonic mean in F1, the geometric mean is dominated by whichever
# value is smaller. A classifier that completely ignores one class will have
# either TPR=0 or TNR=0, making G-Mean = sqrt(1.0 * 0.0) = 0, regardless
# of how well it performs on the other class.
specificity = TN / (TN + FP)
gmean = np.sqrt(recall * specificity)

print('G-Mean computation:')
print(f'  Sensitivity (TPR) = {recall:.4f}')
print(f'  Specificity (TNR) = {specificity:.4f}')
print(f'  G-Mean            = {gmean:.4f}')
print()

# Compare G-Mean across several hypothetical operating points to show how
# it penalises classifiers that trade away performance on one class entirely.
models_gm = [
    ('Ignores minority class',  0.05, 0.99),
    ('Ignores majority class',  0.95, 0.10),
    ('Balanced',                0.80, 0.85),
    ('Our model',               recall, specificity),
    ('Always predicts negative', 0.00, 1.00),
]

print(f'  {"Model":<26} {"Sensitivity":>13} {"Specificity":>13} {"G-Mean":>8}')
print('  ' + '-' * 62)
for name, sens, spec in models_gm:
    gm = np.sqrt(sens * spec)
    print(f'  {name:<26} {sens:>13.3f} {spec:>13.3f} {gm:>8.3f}')

print()
# The "always predicts negative" row makes the key property explicit: a
# classifier that never predicts the minority class achieves perfect
# specificity (it never raises a false alarm on negatives) but zero
# sensitivity, and G-Mean correctly collapses to 0.0.
print('Note: "always predicts negative" achieves Specificity=1.0 but G-Mean=0.0,')
print('because it catches none of the positive class (Sensitivity=0.0).')



---

## 9. Metrics Designed for Imbalanced Data

🔙 [Back to Table of Contents](#table-of-contents)

The metrics covered so far each capture something important, but none of them was specifically designed with class imbalance in mind. This section introduces three metrics that were, each approaching the problem from a different angle.

### 9.1 Balanced Accuracy

Standard accuracy counts every correct prediction equally, which means performance on the majority class dominates the result. Balanced accuracy corrects for this by computing the average recall across both classes, giving each class equal weight regardless of how many instances it contributes.

$$\text{Balanced Accuracy} = \frac{\text{Sensitivity} + \text{Specificity}}{2}$$

For a binary problem this is simply the mean of the true positive rate and the true negative rate. A classifier that always predicts the majority class achieves sensitivity of 0 and specificity of 1, giving a balanced accuracy of exactly 0.5, the same as random guessing, which is the correct verdict. Standard accuracy on the same classifier would report something close to 90% on a 90/10 split, making it look far more capable than it is.

For problems with more than two classes, balanced accuracy generalises to the mean of per-class recall values, one recall score per class.

### 9.2 Cohen's Kappa

Balanced accuracy still measures raw performance. Cohen's Kappa instead asks a subtly different question: **how much better than chance is this classifier?** It corrects observed accuracy for the agreement that would be expected if the classifier were simply assigning labels at random in proportion to the class frequencies.

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

where $p_o$ is the observed accuracy and $p_e$ is the accuracy expected by chance given the class distribution. A kappa of 0 means the classifier is no better than random; a kappa of 1 means perfect agreement with the true labels; negative values indicate performance worse than chance. On imbalanced data, a high-accuracy majority-class classifier will have a kappa close to 0, because most of its correct predictions are exactly what a random classifier would produce anyway.

### 9.3 Matthews Correlation Coefficient (MCC)

The MCC is widely regarded as the most informative single-number summary for binary classification on imbalanced data, because it is the only one of the metrics in this notebook that takes all four cells of the confusion matrix into account simultaneously, without privileging either class.

$$\text{MCC} = \frac{TP \cdot TN - FP \cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$

MCC is a correlation coefficient between the true labels and the predicted labels, ranging from $-1$ to $+1$. A value of $+1$ means perfect prediction; $0$ means the predictions are no better than random; $-1$ means the classifier has inverted every prediction. Unlike accuracy, balanced accuracy, or F1, MCC only reaches a high value when the classifier performs well on all four quadrants of the confusion matrix: high TP and TN, and low FP and FN, simultaneously. It is very difficult to game with an imbalanced dataset.

> 💡 **Intuition:** think of MCC as the metric that asks "if I plotted the true labels against the predicted labels, how strong would the correlation be?" A classifier that gets everything right produces a perfect positive correlation; one that guesses randomly produces no correlation at all.

The code below computes all three of these metrics alongside the metrics from earlier sections, comparing our Section 4.1 classifier against a baseline that always predicts the negative class. The heatmap that follows makes the comparison immediately visual, and the printed summary identifies which metrics are sensitive enough to tell the two classifiers apart.

In [ ]:
# Listing 9.
# ── Imbalance-aware metrics: Balanced Accuracy, MCC, Cohen's Kappa ───────────
#
# The metrics computed so far focus on one or two aspects of classifier
# performance. The three metrics introduced here each attempt to summarise
# overall performance in a single number that remains meaningful even when
# the dataset is heavily imbalanced.
#
# BALANCED ACCURACY: the arithmetic mean of per-class recall, i.e. the
#   average of sensitivity and specificity. Unlike standard accuracy, it
#   gives equal weight to both classes regardless of how many instances
#   each contributes. For binary classification this simplifies to:
#   (Sensitivity + Specificity) / 2
#
# MCC (Matthews Correlation Coefficient): treats the confusion matrix as a
#   contingency table and computes the correlation between the true labels
#   and the predictions. It uses all four cells (TP, TN, FP, FN) and
#   returns a value in [-1, 1], where 1 is a perfect classifier, 0 is
#   no better than random, and -1 is a perfectly inverted classifier.
#   MCC is widely considered the most informative single-number summary
#   for imbalanced binary classification.
#
# COHEN'S KAPPA: measures agreement between two raters (here the classifier
#   and the true labels) corrected for the agreement expected by chance.
#   Kappa = (observed accuracy - chance accuracy) / (1 - chance accuracy).
#   A kappa of 0 means no better than chance; 1 means perfect agreement.
#
# ── Build a baseline "always predicts negative" classifier for comparison ──────
# Predicting the majority class for every instance is the simplest possible
# strategy on an imbalanced dataset. Comparing our real classifier against
# it reveals which metrics are sensitive enough to detect the difference.
y_pred_negative = np.zeros_like(y_te)   # always predicts class 0 (negative)


def _compute_all_metrics(y_true, y_pred, name):
    """
    Compute a full suite of binary classification metrics and return as a dict.

    All rates are derived from the confusion matrix directly rather than
    calling multiple sklearn functions, so the cell counts are consistent
    with the TP, TN, FP, FN variables used throughout this notebook.
    """
    cm             = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1   = 2 * prec * sens / (prec + sens) if (prec + sens) > 0 else 0.0
    gmn  = np.sqrt(sens * spec)

    return {
        'Model':        name,
        'Accuracy':     accuracy_score(y_true, y_pred),
        'Recall':       sens,
        'Precision':    prec,
        'Specificity':  spec,
        'F1':           f1,
        'G-Mean':       gmn,
        'Balanced Acc': balanced_accuracy_score(y_true, y_pred),
        'MCC':          matthews_corrcoef(y_true, y_pred),
        "Cohen's K":    cohen_kappa_score(y_true, y_pred),
    }


results_all = pd.DataFrame([
    _compute_all_metrics(y_te, y_pred,          'Our classifier'),
    _compute_all_metrics(y_te, y_pred_negative, 'Always predicts negative'),
]).set_index('Model')

print('Full metric comparison:')
print(results_all.round(3).to_string())
print()

# ── Heatmap: all metrics side by side ────────────────────────────────────────
# A diverging Red-Yellow-Green colourmap makes it immediately obvious which
# cells are strong (green) and which are weak (red). vmin=0 and vmax=1
# keep MCC and Cohen's Kappa on a comparable scale to the rate-based metrics,
# noting that both can take negative values in adversarial cases: clamping
# them at 0 in the colour scale is a display choice, not a data transformation.
fig, ax = plt.subplots(figsize=(10, 3))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

sns.heatmap(
    results_all.astype(float),
    annot=True, fmt='.3f',
    cmap='RdYlGn', vmin=0, vmax=1,
    linewidths=0.5, linecolor='white',
    ax=ax, annot_kws={'size': 10},
)
ax.set_title(
    'Figure 6: all metrics side by side — which ones expose the always-negative classifier?',
    fontsize=11,
)
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

# ── Which metrics successfully distinguish the two classifiers? ───────────────
# A metric that returns a similar value for both models is not useful for
# evaluating classifiers on imbalanced data. A difference threshold of 0.05
# is used here as a practical lower bound on "meaningfully different".
print('Metrics that distinguish our classifier from the always-negative baseline')
print('(difference > 0.05):')
for col in results_all.columns:
    diff = (results_all.loc['Our classifier', col]
            - results_all.loc['Always predicts negative', col])
    if abs(diff) > 0.05:
        print(f'  {col:<14}: ours = {results_all.loc["Our classifier", col]:.3f}, '
              f'baseline = {results_all.loc["Always predicts negative", col]:.3f}  '
              f'({"+" if diff > 0 else ""}{diff:.3f})')

---

The printed summary tells us which metrics are actually doing useful work here. Accuracy, recall, specificity, and balanced accuracy all fail to distinguish the two classifiers by more than 0.05, which is a damning result: those metrics cannot tell apart a real trained classifier from one that never predicts the positive class at all.

The five metrics that do show a meaningful difference are worth examining individually. Precision registers the largest gap at +0.429, because the always-negative baseline makes no positive predictions whatsoever, giving it a precision of exactly 0 by definition. This makes precision look like a strong discriminator here, but it is somewhat misleading: the gap exists largely because the baseline's denominator is zero rather than because our classifier's precision is genuinely high at 0.429.

F1 shows a gap of only +0.076, reflecting the fact that our classifier's very low recall drags the harmonic mean down close to zero even though its precision is reasonable. G-Mean shows a slightly larger gap at +0.203, for the same structural reason: a sensitivity of 0.042 pulls the geometric mean down hard, but not quite as far as the harmonic mean in F1, because the geometric mean is somewhat less severe a penalty than the harmonic mean at these extreme values.

MCC and Cohen's Kappa both show small but non-zero gaps of +0.103 and +0.056. These are the most honest summary of what is happening: our classifier is better than always predicting negative, but only marginally so at the default threshold on this imbalanced dataset. Both metrics are designed to correct for the class distribution, so they do not inflate the result the way raw accuracy does, and the small values correctly reflect that the improvement over the baseline is real but modest.

The broader lesson from this comparison is that on an imbalanced dataset, the choice of metric is important. Accuracy, recall, and specificity all told us very little here. The metrics that revealed the true picture were those that either consider both classes simultaneously (balanced accuracy, G-Mean, MCC) or that correct for chance agreement (Cohen's Kappa).


---

## 10. Putting It All Together: A Binary Classification Evaluation Workflow

🔙 [Back to Table of Contents](#table-of-contents)

The sections above introduced each metric individually. This section shows how to assemble them into a single, reusable evaluation workflow: one that you can drop any trained classifier into and get a complete picture of its performance in one pass.

To keep the focus on the process rather than the problem, we use a fresh synthetic dataset with a 1:4 class imbalance and deliberate overlap between the two classes, so no classifier will look artificially good. The classifier we'll use is Naive Bayes, chosen purely for simplicity: it is fast, has no hyperparameters to tune, and its performance is modest enough on overlapping data that the metrics tell an interesting story. The workflow is identical for any other classifier, and the section ends by showing exactly what you would change to swap one in.

### 10.1 Building the dataset

The dataset is generated with the `make_classification` function, using the same approach as Section 4.1, with three adjustments: a 1:4 class ratio, a larger sample count to give the metrics more stability, and a higher degree of feature overlap to make the classification problem genuinely difficult. Run the code below to create the new dataset.

In [ ]:
# Listing 10.
# ── Dataset ───────────────────────────────────────────────────────────────────
# 1:4 imbalance: 20% positive (minority), 80% negative (majority).
# n_clusters_per_class=3 and flip_y=0.08 increase class overlap, making
# the classification problem genuinely difficult and the metrics more
# informative than on a clean, separable problem.
POSITIVE_FRACTION_NEW = 0.20
N_SAMPLES_NEW         = 3_000
RANDOM_STATE_NEW      = 7

X_new, y_new = make_classification(
    n_samples=N_SAMPLES_NEW,
    n_features=8,
    n_informative=5,
    n_redundant=2,
    n_clusters_per_class=3,
    weights=[1 - POSITIVE_FRACTION_NEW, POSITIVE_FRACTION_NEW],
    flip_y=0.08,
    random_state=RANDOM_STATE_NEW,
)

Next we create our training and test splits, and then scale the data too.

In [ ]:
# Listing 11.
X_tr_new, X_te_new, y_tr_new, y_te_new = train_test_split(
    X_new, y_new,
    test_size=0.30,
    stratify=y_new,         # preserve the 1:4 ratio in both splits
    random_state=RANDOM_STATE_NEW,
)

scaler_new = StandardScaler()
X_tr_new_s = scaler_new.fit_transform(X_tr_new)
X_te_new_s = scaler_new.transform(X_te_new)   # fit on train only

print('Dataset summary')
print(f'  Training set : {len(y_tr_new):,} samples  '
      f'({y_tr_new.sum():,} positive, {(y_tr_new == 0).sum():,} negative)')
print(f'  Test set     : {len(y_te_new):,} samples  '
      f'({y_te_new.sum():,} positive, {(y_te_new == 0).sum():,} negative)')
print()

### 10.2 Training the classifier

Naive Bayes assumes that features are conditionally independent given the class label, which is rarely true in practice but makes the model very fast and easy to reason about. `GaussianNB` is the continuous-feature variant, appropriate here because our synthetic features are real-valued. It requires no hyperparameter tuning and no iteration limit, so the training call is a single line.

To use a different classifier, replace these three lines. For example, to use a random forest instead:

```python
from sklearn.ensemble import RandomForestClassifier

clf_nb = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
clf_nb.fit(X_tr_new_s, y_tr_new)
y_pred_nb = clf_nb.predict(X_te_new_s)
```

Run the code below to train and test the model.

In [ ]:
# Listing 12.
# ── Classifier ────────────────────────────────────────────────────────────────
# GaussianNB is the continuous-feature variant of Naive Bayes, appropriate
# for real-valued features. It requires no hyperparameter tuning and no
# iteration limit, so the training call is a single line.
# To swap in a different classifier, replace only these three lines:
#
#   from sklearn.ensemble import RandomForestClassifier
#   clf_nb = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE_NEW)
#   clf_nb.fit(X_tr_new_s, y_tr_new)
#
# Everything below is identical regardless of which classifier is used.
clf_nb     = GaussianNB()
clf_nb.fit(X_tr_new_s, y_tr_new)
y_pred_nb  = clf_nb.predict(X_te_new_s)

### 10.3 Computing and reporting the metrics

With predictions made above, the evaluation workflow is always the same three steps:

1. extract the confusion matrix counts.
2. pass them through the metric functions.
3. display the results.

Run the code below to create the matrix and show it. Note that the `ravel()` function used below, flattens a multi-dimensional array (like a 2D confusion matrix) into a 1D array. In the code, `cm_nb.ravel()` converts the 2x2 confusion matrix into a flat list `[tn, fp, fn, tp]`, which is then unpacked into the four variables.

In [ ]:
# Listing 13.
# ── Confusion matrix ──────────────────────────────────────────────────────────
cm_nb                        = confusion_matrix(y_te_new, y_pred_nb)
tn_nb, fp_nb, fn_nb, tp_nb   = cm_nb.ravel()

fig, ax = plt.subplots(figsize=(5, 4))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

ConfusionMatrixDisplay(
    cm_nb, display_labels=['Negative', 'Positive'],
).plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title('Confusion matrix: Naive Bayes (test set)')
plt.tight_layout()
plt.show()

Finally, we compute ALL the metrics directly.

In [ ]:
# Listing 14.
# ── Metrics ───────────────────────────────────────────────────────────────────
# Sensitivity and specificity are derived from the confusion matrix counts
# directly for consistency with the rest of this notebook, then passed into
# G-Mean. All other metrics use sklearn functions for conciseness and to
# cross-check the manual computations above.
sens_nb = tp_nb / (tp_nb + fn_nb) if (tp_nb + fn_nb) > 0 else 0.0
spec_nb = tn_nb / (tn_nb + fp_nb) if (tn_nb + fp_nb) > 0 else 0.0

metrics_nb = {
    'Accuracy':       accuracy_score(y_te_new, y_pred_nb),
    'Balanced Acc':   balanced_accuracy_score(y_te_new, y_pred_nb),
    'Precision':      precision_score(y_te_new, y_pred_nb, zero_division=0),
    'Recall (TPR)':   sens_nb,
    'Specificity':    spec_nb,
    'F1':             f1_score(y_te_new, y_pred_nb, zero_division=0),
    'G-Mean':         np.sqrt(sens_nb * spec_nb),
    'MCC':            matthews_corrcoef(y_te_new, y_pred_nb),
    "Cohen's Kappa":  cohen_kappa_score(y_te_new, y_pred_nb),
}

results_nb = pd.DataFrame(metrics_nb, index=['Naive Bayes']).T
results_nb.columns = ['Score']

print('Full metric report: Naive Bayes')
print(results_nb.round(3).to_string())

---

**What the results show**

The confusion matrix shows 651 true negatives, 70 true positives, 48 false positives, and 131 false negatives from a test set of 900 samples. The first thing to notice is the asymmetry between FN and FP: the classifier missed 131 actual positives but only raised 48 false alarms. It is being too conservative about predicting the positive class, which is the expected behaviour of Naive Bayes on an imbalanced dataset at the default threshold.

Accuracy reports 0.801, which sounds reasonable until you look at what is driving it. The vast majority of the test set is negative, and the classifier gets most of those right (651 out of 699), so accuracy is largely a reflection of performance on the easy majority class. Balanced accuracy corrects for this immediately: at 0.640, it reflects the mean of specificity (0.931) and recall (0.348), giving equal weight to both classes, and the result is considerably less flattering.

The precision-recall split tells the clearest story about the classifier's behaviour. Precision of 0.593 means that when the classifier does predict positive, it is right about 59% of the time, which is reasonable. But recall of 0.348 means it only finds around one in three actual positives, missing the other two thirds entirely. The classifier is selective to a fault: its positive predictions are moderately reliable, but it makes far too few of them.

F1 at 0.439 captures that tension by penalising the low recall through the harmonic mean. G-Mean at 0.570 is somewhat more generous because it combines recall with the high specificity of 0.931 using the geometric mean, which is a less severe penalty at these values. The two metrics are not contradicting each other: they are measuring different things, and the gap between them (0.439 vs 0.570) is itself informative. It tells you that the classifier performs much better on the negative class than on the positive class, which is exactly what the raw counts show.

MCC at 0.345 and Cohen's Kappa at 0.328 are the most conservative summary figures. Both correct for the class distribution and for chance agreement respectively, and both land in a similar range, which gives some confidence that the estimate is stable. A value around 0.33 on both measures means the classifier is genuinely better than chance, but not by a large margin, and the improvement over a naive always-negative baseline is real but modest. These are honest numbers for a simple classifier on a problem with deliberate overlap and label noise.

---

## 11. Multi-Class Metrics: Macro, Micro, and Weighted Averaging

🔙 [Back to Table of Contents](#table-of-contents)

Everything covered in this notebook so far has assumed binary classification: one positive class and one negative class. When there are more than two classes, precision, recall, and F1 are still defined, but they need to be computed per class first and then combined into a single summary figure. The way that combination is done matters enormously, and the three strategies available, macro, micro, and weighted averaging, can produce very different numbers from the same set of predictions.

The per-class computation works by treating each class in turn as the "positive" class and grouping all other classes together as "negative", the same one-vs-rest logic used throughout whenever we deal with multi-class problems. This gives one precision, one recall, and one F1 score per class. The question is then how to average them.

| Strategy | How it works | Use when |
|---|---|---|
| **Macro** | Compute the metric for each class independently, then take the simple (unweighted) average | All classes are equally important regardless of how many instances each has |
| **Micro** | Pool the TP, FP, and FN counts across all classes, then compute the metric once from the totals | Larger classes should contribute more to the final score |
| **Weighted** | Compute the metric per class, then average weighted by each class's share of the total test instances (its support) | You want overall performance but with class size taken into account |

The choice between these is not merely technical: it is a statement about what you care about. On an imbalanced dataset, macro averaging treats a rare class with 50 instances the same as a common class with 5,000 instances, which means poor performance on the majority class is fully exposed. Micro averaging does the opposite: it allows the majority class to dominate, which can make the summary figure look strong even when the minority class is being handled poorly. Weighted averaging sits between the two but still tends to favour larger classes.

> ⚠️ **Warning:** on an imbalanced dataset, micro-averaged F1 is numerically close to accuracy and shares the same weakness: a classifier that ignores the minority class entirely can still report a high micro F1. If minority class performance matters, macro or weighted averaging, combined with a per-class breakdown, gives a more honest picture.

Using scikit-learn's `classification_report`, we can obtain per-class precision, recall, and F1 alongside all three averages in a single call, making it straightforward to see where they diverge. You can even do this for a binary classification problem, e.g.,

```python
from sklearn.metrics import classification_report

print(classification_report(
    y_te_new, y_pred_nb,
    target_names=['Negative', 'Positive'],
    digits=3,
))
```

The `classification_report` output is the standard starting point for any multi-class evaluation, though in the cell right below this one, I use it for the binary case. Reading it column by column rather than row by row is usually more informative: compare the precision column across classes, then the recall column, before looking at the aggregated rows at the bottom. If the macro and weighted F1 figures diverge noticeably, that divergence is almost always a signal that class performance is uneven and worth investigating further.

In [ ]:
# Listing 15.
from sklearn.metrics import classification_report

print(classification_report(
    y_te_new, y_pred_nb,
    target_names=['Negative', 'Positive'],
    digits=3,
))

---

### 11.1 Putting It All Together: A Multi-class Classification Evaluation Workflow

To show you how we collect metrics for a multi-class task, we need a multi-class problem. The Iris dataset which we've used before is ideal for this purpose: it has three balanced classes, a well-understood structure, and is small enough that every number in the output can be traced back to the confusion matrix by hand. The classifier we'll use this time will be a shallow decision tree, chosen because it is fast, interpretable, and produces a realistic mix of correct and incorrect predictions on Iris rather than near-perfect accuracy.

The workflow below is the same one you would apply to any multi-class problem. Only the dataset, the classifier, and the class names would change.

**Step 1.** First, we load the data.

In [ ]:
# Listing 16.
# ── Dataset ───────────────────────────────────────────────────────────────────
# Iris is balanced (50 instances per class), which means macro, micro, and
# weighted averages will be close to each other here. That is intentional:
# understanding what each average measures on balanced data makes it easier
# to see what changes when the classes become imbalanced.
iris = load_iris()
X_ir, y_ir = iris.data, iris.target
class_names = iris.target_names   # ['setosa', 'versicolor', 'virginica']

**Step 2.** Next we split the data into training and test sets, then scale the data.

In [ ]:
# Listing 17.
X_tr_ir, X_te_ir, y_tr_ir, y_te_ir = train_test_split(
    X_ir, y_ir,
    test_size=0.30,
    stratify=y_ir,
    random_state=0,
)

sc_ir = StandardScaler()
X_tr_ir_s = sc_ir.fit_transform(X_tr_ir)
X_te_ir_s = sc_ir.transform(X_te_ir)

**Step 3.** The next step is to train and test the classifier.

In [ ]:
# Listing 18.
# ── Classifier ────────────────────────────────────────────────────────────────
# max_depth=4 gives the tree enough capacity to learn the Iris boundaries
# without overfitting to the point where the test metrics become trivial.
clf_ir    = DecisionTreeClassifier(max_depth=4, random_state=0)
clf_ir.fit(X_tr_ir_s, y_tr_ir)
y_pred_ir = clf_ir.predict(X_te_ir_s)

**Step 4.** With predictions ready, the next step involves creating the confusion matrix. For a three-class problem the matrix is 3×3: each row is a true class and each column is a predicted class, with the diagonal holding the correctly classified instances.

In [ ]:
# Listing 19.
# ── Per-class recall from the confusion matrix ────────────────────────────────
# For each class i, the diagonal entry cm[i, i] is the number of instances
# of that class that were correctly predicted (TP for that class). The row
# sum cm[i, :] is the total number of actual instances of class i, so
# cm[i, :].sum() - cm[i, i] gives the false negatives for that class.
cm_ir = confusion_matrix(y_te_ir, y_pred_ir)

print('Per-class recall (computed manually from the confusion matrix):')
for i, name in enumerate(class_names):
    tp_i = cm_ir[i, i]
    fn_i = cm_ir[i, :].sum() - tp_i
    r_i  = tp_i / (tp_i + fn_i)
    print(f'  {name:<12}: TP={tp_i}, FN={fn_i}, Recall={r_i:.4f}')
print()

**Step 5.** The per-class recall values are the raw ingredients for all three averaging strategies. Once you can read them from the confusion matrix directly, the aggregated numbers from scikit-learn become much easier to interpret.

In [ ]:
# Listing 20.
# ── Aggregated metrics: macro, micro, weighted ────────────────────────────────
# Each averaging strategy is computed by passing a different string to the
# average= argument. Calling all three in a loop makes the differences
# immediately visible side by side.
print('Aggregated metrics:')
for avg in ['macro', 'micro', 'weighted']:
    p = precision_score(y_te_ir, y_pred_ir, average=avg)
    r = recall_score(y_te_ir,    y_pred_ir, average=avg)
    f = f1_score(y_te_ir,        y_pred_ir, average=avg)
    print(f'  {avg:<10}  P={p:.3f}  R={r:.3f}  F1={f:.3f}')
print()

**Step 6.** Next we use scikit-learn's `classification_report` produces all of the above in a single call, with per-class rows followed by the three aggregated rows at the bottom. It is the standard output to include in any evaluation report.

In [ ]:
# Listing 21.
print(classification_report(
    y_te_ir, y_pred_ir,
    target_names=class_names,
    digits=3,
))

When reading a classification report, work column by column rather than row by row. First compare precision down the column: are there classes the model is unreliable on? Then compare recall: are there classes the model is systematically missing? Only then look at the aggregated rows. If macro and weighted F1 diverge noticeably, it is almost always a sign that class sizes are uneven and that one or more minority classes are being handled poorly.

**Step 7.** Next we create the confusion matrix plots.

In [ ]:
# Listing 22.
# ── Figure 7: confusion matrix and per-class bar chart ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

# Left panel: confusion matrix as a heatmap. fmt='d' prints integer counts
# rather than floats, which are easier to read for small cell values.
ax = axes[0]
sns.heatmap(
    cm_ir, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names,
    linewidths=0.5, linecolor='white',
    ax=ax, annot_kws={'size': 14},
)
ax.set_title(
    f'Multi-class confusion matrix (Iris)\n'
    f'Accuracy = {accuracy_score(y_te_ir, y_pred_ir):.1%}',
)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')

# Right panel: per-class precision, recall, and F1 as grouped bars.
# average=None tells sklearn to return one value per class rather than
# a single aggregated figure.
metrics_mc = {
    name: {
        'Precision': precision_score(y_te_ir, y_pred_ir, average=None)[i],
        'Recall':    recall_score(y_te_ir,    y_pred_ir, average=None)[i],
        'F1':        f1_score(y_te_ir,        y_pred_ir, average=None)[i],
    }
    for i, name in enumerate(class_names)
}
df_mc = pd.DataFrame(metrics_mc).T

ax = axes[1]
x_pos = np.arange(len(class_names))
width = 0.25
for j, (metric, col) in enumerate(zip(
    ['Precision', 'Recall', 'F1'],
    ['steelblue', 'tomato', 'seagreen'],
)):
    ax.bar(x_pos + j * width, df_mc[metric], width=width,
           color=col, edgecolor='white', lw=0.4, label=metric, alpha=0.85)

ax.set_xticks(x_pos + width)
ax.set_xticklabels(class_names)
ax.set_ylabel('Score')
ax.set_title('Per-class Precision, Recall, and F1 (Iris)')
ax.legend(fontsize=9)
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(
    'Figure 7: Multi-class evaluation — confusion matrix and per-class metrics',
    fontsize=11,
)
plt.tight_layout()
plt.show()

---
> 📌 **What to notice:** on a balanced dataset like Iris, macro and weighted F1 are nearly identical because each class contributes the same number of test instances. The bar chart on the right makes it easy to see whether any single class is dragging down the overall score. If you were to repeat this workflow on an imbalanced multi-class dataset, the macro and weighted rows in the classification report would diverge, and that divergence is the first thing to investigate.



---

## 12. Bias and Variance: Two Sources of Error

🔙 [Back to Table of Contents](#table-of-contents)

The metrics covered in this notebook measure *how much* a model is wrong. Bias and variance explain *why* it is wrong. Every prediction error a model makes can be decomposed into three components:

$$\text{Total Error} = \underbrace{\text{Bias}^2}_{\text{from wrong assumptions}} + \underbrace{\text{Variance}}_{\text{from sensitivity to data}} + \underbrace{\text{Irreducible noise}}_{\text{cannot be eliminated}}$$

* **Bias** is the error that comes from making overly simple assumptions about the data. A high-bias model lacks the capacity to represent the true underlying pattern: it imposes a structure that is too rigid, and as a result it is systematically wrong on both the training data and any new data it sees. No amount of additional training data will fix a model whose fundamental form is wrong. This is what we mean by **underfitting**.

* **Variance** is the error that comes from being too sensitive to the particular training sample the model was fitted on. A high-variance model has enough capacity not just to learn the true pattern but also to memorise the random noise in the training data. Show it a slightly different training set and it will produce a noticeably different model, because it is fitting the noise rather than the signal. This is **overfitting**: the model performs well on training data and poorly on anything else.

The tension between the two is unavoidable. Increasing a model's complexity reduces bias, because a more expressive model can approximate a wider range of true patterns, but it increases variance, because a more expressive model also has more room to fit noise. Decreasing complexity does the reverse. Finding the right point on that spectrum is one of the central practical challenges of machine learning, and it is precisely what cross-validation (covered in Notebook 10) is designed to help with.

> 💡 **Intuition:** bias is like a scale that is always 2 kg too high: it gives the wrong answer consistently, and recalibrating it (getting more data) will not help if the mechanism is fundamentally broken. Variance is like a scale that gives a different reading every time you step on it: the average might be right, but any individual reading is unreliable.

---

Figure 8 below illustrates both failure modes side by side using polynomial regression on a small synthetic dataset. Before looking at the figure, it is worth briefly explaining what polynomial regression is and why it is useful for illustrating bias and variance.

A **polynomial** is an expression built from powers of a variable: $x$, $x^2$, $x^3$, and so on. A degree-1 polynomial is a straight line ($y = ax + b$); a degree-2 polynomial is a parabola ($y = ax^2 + bx + c$); higher degrees produce curves with more bends. **Polynomial regression** fits a curve of a chosen degree to data by finding the coefficients that minimise the prediction error, using exactly the same least-squares idea as linear regression from Notebook 5. The degree is a hyperparameter that controls how flexible the fitted curve is.

This makes polynomial regression a natural tool for demonstrating bias and variance, because the degree directly controls the model's capacity. The data in Figure 8 follows a gently curved sinusoidal pattern with added noise: the true shape is not a straight line, so a degree-1 polynomial (a straight line) simply cannot follow it no matter how well it is fitted. It will always cut through the middle of the curve, missing the peaks and troughs systematically. That is bias made visible. At the other extreme, a degree-20 polynomial has enough bends available that it can weave through almost every individual training point, including the noisy ones that deviate from the true pattern. It fits the data it has seen almost perfectly but produces a wildly contorted curve that bears little resemblance to the true function. That is variance made visible. The degree-4 polynomial sits between the two: flexible enough to approximate the underlying shape, but not so flexible that it chases noise.

Because the dataset is synthetic, the true underlying function is known and is shown as a dashed line in each panel. This means each fitted curve can be compared directly against ground truth rather than just against held-out test error, making it clear at a glance whether the model is systematically wrong (bias) or erratically right on training data and wrong elsewhere (variance).

In [ ]:
# Listing 23.
%matplotlib widget
from visualisations.Figure_8 import show
show()

---

The polynomial regression example in Figure 8 makes bias and variance visible as curve shapes. Figure 9 below offers a different perspective on the same idea that is often easier to hold in your head: the dartboard analogy.

Imagine training the same model architecture 40 times, each time on a different random sample drawn from the same population. Each trained model makes a prediction at a fixed test point, represented as a single dot on the board. The bullseye (red cross) is the true value at that point. The gold star is the mean of all 40 predictions. With that setup, two things become immediately readable from the scatter of dots:

- **Bias** is the distance between the gold star and the bullseye. If the average prediction is close to the true value, bias is low. If it is consistently off to one side, bias is high, regardless of how tightly the dots cluster.
- **Variance** is the spread of the dots around the gold star. If all 40 predictions land in roughly the same place, variance is low. If they scatter across the board, variance is high, regardless of where that scatter is centred.

Figure 9 shows all four combinations of high and low bias and variance side by side. The top-left panel (low bias, low variance) is the ideal: a tight cluster centred on the bullseye. The top-right (low bias, high variance) shows what happens when the model is correct on average but unreliable for any individual prediction. The bottom-left (high bias, low variance) shows a model that is consistently wrong in the same direction, confidently making the same mistake every time. The bottom-right (high bias, high variance) is the worst case: predictions that are both off-centre and scattered, wrong and inconsistent simultaneously.

In [ ]:
# Listing 24.
%matplotlib widget
from visualisations.Figure_9 import show
show()

---

The dartboard analogy in Figure 9 shows bias and variance as properties of a model family: what happens across many training runs. Figure 10 makes a different but complementary point: as you move along the complexity axis, one of these properties improves while the other gets worse, and the trade-off is directly measurable in terms of training and test error.

To see this concretely, Figure 10 uses a decision tree classifier as a stand-in for "model complexity", because decision trees have an easily tunable parameter, `max_depth`, that controls complexity in a particularly transparent way. Recall that a decision tree works by repeatedly splitting data on the feature that best separates the classes at each step, forming a branching structure of yes/no decisions. The `max_depth` parameter sets a hard limit on how many splits deep the tree can go: a depth of 1 means a single split producing two regions; a depth of 20 means up to $2^{20}$ possible regions, more than enough to carve the training data into arbitrarily small pieces. Increasing `max_depth` is therefore a clean, monotonic dial for complexity, which is exactly what is needed to trace out the bias-variance curve.

Figure 10 sweeps `max_depth` from 1 to 20. At each depth, the same tree is trained on the same training set and evaluated on both the training set and a held-out test set. The error rate at each depth is then plotted as a curve.

Two things happen as depth increases. Training error falls monotonically, because a deeper tree can always partition the training data more finely and reduce its own mistakes. There is nothing surprising about this: given enough depth, a decision tree will eventually fit every training point perfectly and report zero training error. Test error tells a different story. It falls initially as the tree gains enough capacity to capture the true pattern in the data, but then it starts to rise again as the tree begins fitting the noise specific to the training sample rather than the underlying signal. The minimum of the test error curve is the point of optimal complexity, the depth at which bias has been reduced enough without variance having grown too large.

The shaded region between the two curves is the generalisation gap: the difference between how well the model performs on data it was trained on and how well it performs on data it has never seen. At low depths the gap is small because both errors are high for the same reason (the model is too simple). At high depths the gap widens because training error continues to fall while test error rises, which is the signature of overfitting.


In [ ]:
# Listing 25.
%matplotlib widget
from visualisations.Figure_10 import show
show()


The three figures in this section tell a consistent story from three different angles. Figure 8 shows what underfitting and overfitting look like in the shape of a fitted curve. Figure 9 shows what they mean statistically, across many training runs, in terms of the spread and position of predictions. Figure 10 shows what they cost in practice, as a measurable gap between training and test error that widens as complexity grows.

The key insight that unites all three is that neither extreme is desirable, and neither is avoidable by simply collecting more data or trying harder. A model that is too simple will be wrong in the same way regardless of how much data it sees, because the problem is its form, not its fit. A model that is too complex will always find something in the training data to memorise, because the problem is its capacity, not its data. The only solution is to find the right level of complexity for the problem at hand, which requires being able to diagnose which failure mode you are in.

That is the subject of the next section.

---

## 13. Diagnosing Underfitting and Overfitting

🔙 [Back to Table of Contents](#table-of-contents)

The bias-variance trade-off produces concrete, observable symptoms in your model's behaviour that you can learn to recognise and act on. The distinction between the two failure modes is almost always visible before you even look at a test set, if you know what to look for.

### 13.1 Diagnosing underfitting (high bias)

The defining symptom of underfitting is poor performance on **both** training and test data. If a model cannot even fit the data it was trained on, it has not learned the underlying pattern at all, and there is no reason to expect it to do any better on new data. In practical terms this looks like:

- Accuracy, F1, recall, and precision are all low on the training set, not just the test set.
- The model may predict only the majority class on every instance, or produce a decision boundary that is far too simple to separate the classes.
- Training and test error are both high, and the generalisation gap (the difference between them) is small, because the model is failing equally on both.
- Collecting more training data makes no difference. The model is not struggling because it has seen too few examples; it is struggling because it lacks the capacity to represent the true pattern, and no quantity of additional data changes that.

The fixes for underfitting all involve giving the model more capacity or better information: increasing `max_depth` on a decision tree, adding polynomial features, or switching to a more expressive model architecture entirely.

### 13.2 Diagnosing overfitting (high variance)

Overfitting produces the opposite symptom: very low error on the training set and noticeably higher error on the test set. The model has learned the training data well, but what it has learned includes the noise rather than just the signal. In practical terms:

- Training accuracy is high, sometimes near perfect, while test accuracy is meaningfully lower. This gap is the key diagnostic signal.
- Decision boundaries are irregular or contorted, appearing to have been drawn around individual training points rather than around the true class regions.
- The model is sensitive to the specific training sample: retrain it on a slightly different set of data and the predictions change noticeably, sometimes dramatically.
- Collecting more training data does help, because more data makes it harder for the model to memorise individual points and forces it to generalise. This is one of the few reliable fixes for overfitting that does not require changing the model itself.

Other fixes include reducing model complexity (lowering `max_depth`), using dropout or early stopping in neural networks, and using cross-validation to detect the generalisation gap during model selection rather than discovering it only after deployment.

### 13.3 Reading the generalisation gap

In practice, the most reliable diagnostic is simply to compare training and test error directly. A useful rule of thumb:

| Observation | Likely diagnosis |
|---|---|
| Both training and test error high, small gap | Underfitting (high bias) |
| Training error low, test error much higher, large gap | Overfitting (high variance) |
| Both training and test error low, small gap | Good fit |
| Both training and test error low, but test error rises with more data | Mild overfitting |

> ⚠️ **Warning:** a small generalisation gap does not automatically mean a good model. If both training and test error are high and close together, the model is underfitting, and the small gap is not a sign of health, it is a sign that the model has failed equally on both sets. Always check the absolute level of the errors, not just the gap between them.


---
## 14. Regularisation: Controlling Complexity

🔙 [Back to Table of Contents](#table-of-contents)

The previous section showed that increasing model complexity eventually leads to overfitting: the model gains enough capacity to fit the noise in the training data rather than just the signal. One approach to this problem is to simply choose a simpler model. But there is a more flexible solution that allows you to keep a complex model while discouraging it from using that complexity excessively: **regularisation**.

Regularisation works by adding an extra term to the loss function, the quantity the model minimises during training, that penalises the model for having large coefficient values:

$$\text{Loss}_{\text{regularised}} = \underbrace{\text{Original loss}}_{\text{fit to training data}} + \underbrace{\lambda \cdot \text{Penalty}}_{\text{cost of complexity}}$$

To understand why penalising large coefficients controls complexity, it helps to think about what large coefficients actually do. A linear model with very large weights can produce very steep, sharply curving decision boundaries: small changes in the input produce large swings in the output, which is exactly the behaviour that allows a model to weave through every training point including its noise. By adding a cost to having large weights, regularisation pushes the optimiser to find a solution that is both a good fit to the training data and has small, well-behaved coefficients. The result is a smoother model that is less likely to have memorised the training data's specific quirks.

The parameter $\lambda$ (lambda) controls how strongly the penalty is applied. A large $\lambda$ means the penalty term dominates and the model is forced toward small coefficients even if that means fitting the training data less precisely. A small $\lambda$ means the original training loss dominates and the model is free to fit the data closely. Scikit-learn expresses this through the parameter $C = 1/\lambda$: a large $C$ means weak regularisation (the model can fit freely), and a small $C$ means strong regularisation (the model is constrained).

| $\lambda$ value | $C$ value | Effect |
|---|---|---|
| $\lambda = 0$ | $C \to \infty$ | No regularisation: model can overfit freely |
| $\lambda$ small | $C$ large | Mild regularisation: slight smoothing effect |
| $\lambda$ large | $C$ small | Strong regularisation: model may underfit |

Choosing the right value of $\lambda$ is itself a **hyperparameter tuning** problem, since the optimal strength depends on the dataset. The standard approach is cross-validation: try a range of $\lambda$ values, evaluate each on held-out validation folds, and select the one that produces the best generalisation performance.

### 14.1 L2 (Ridge) Regularisation

There are two widely used choices of penalty term, and the choice between them affects not just the strength of regularisation but the shape of the solution it produces.

The **L2 penalty**, also known as Ridge regularisation, uses the sum of the squared coefficients:

$$\text{L2 penalty} = \lambda \sum_j w_j^2$$

Squaring the coefficients means that large weights are penalised disproportionately more than small ones: a coefficient of 4 contributes 16 to the penalty, while a coefficient of 1 contributes only 1. This encourages the optimiser to spread weight across many features rather than concentrating it on a few, and it shrinks all coefficients steadily towards zero as $\lambda$ increases. Crucially, L2 regularisation rarely drives any individual coefficient all the way to exactly zero: it suppresses them but keeps them in the model. This is why Ridge is sometimes described as producing a "dense" solution.

> 💡 **Intuition:** L2 regularisation behaves like a rubber band attached to each coefficient, pulling it toward zero with a force proportional to its current size. A small coefficient barely feels the pull; a large coefficient is pulled back hard.

---

The code below creates Figure 11. It shows four panels, each fitting the same degree-15 polynomial to the same 30 training points, with only the regularisation strength $\lambda$ changing between them. At $\lambda = 0.0001$ the penalty is so weak that it has almost no effect: the curve oscillates between training points, fitting the noise as faithfully as the signal, and the coefficient norm $\|w\|$ is very large.

As $\lambda$ increases to 0.1 and then 10, the oscillations progressively dampen: the penalty is forcing the coefficients toward zero, which removes the model's ability to make sharp local turns, and the fitted curve increasingly resembles the true underlying function shown as a dashed line. At $\lambda = 1000$ the penalty is strong enough to suppress the coefficients to near zero, and the curve loses detail: it can no longer follow even the genuine curvature of the data, and the model has moved from overfitting into underfitting.

The train RMSE reported in each panel title tells a consistent story with Figure 10: it rises as $\lambda$ increases, because a more constrained model fits its training data less precisely. The coefficient norm $\|w\|$ tells the complementary story: it shrinks steadily as $\lambda$ increases, which is the direct mechanism producing the smoothing. The practical takeaway is that regularisation does not change the model's architecture or its degree: it is still a degree-15 polynomial in every panel. What it changes is how much of that capacity the model is permitted to use.

In [ ]:
# Listing 26.
%matplotlib widget
from visualisations.Figure_11 import show
show()

---

### 14.2 L1 (Lasso) Regularisation

Ridge regularisation shrinks all coefficients toward zero but rarely eliminates any of them entirely. This is useful when you believe most features contribute something to the prediction. But in many real problems, only a small number of features are genuinely informative and the rest are noise. Keeping all of them in the model, even with small coefficients, wastes capacity and can hurt generalisation. A different penalty, **Lasso** (Least Absolute Shrinkage and Selection Operator), addresses this directly.

The Lasso penalty replaces the squared coefficients of Ridge with their absolute values:

$$\text{L1 penalty} = \lambda \sum_j |w_j|$$

This seemingly small change in the penalty's shape has a significant practical consequence. The L2 penalty grows quadratically: a coefficient of 2 contributes four times as much penalty as a coefficient of 1, so the optimiser has a strong incentive to reduce large coefficients but only a weak incentive to reduce small ones. The L1 penalty grows linearly: a coefficient of 2 contributes exactly twice as much as a coefficient of 1, which means the pull toward zero is constant regardless of how small a coefficient already is. This makes it geometrically efficient for the optimiser to push small coefficients all the way to exactly zero rather than leaving them hovering near-zero as Ridge does.

The practical consequence is that Lasso performs **automatic feature selection**. Features whose coefficients are driven to zero are effectively removed from the model entirely, and what remains is a **sparse** model that uses only a subset of the available features.

> 💡 **Intuition:** Ridge is like dimming every light in a room to save power: everything stays on but at reduced brightness. Lasso is like deciding which lights to switch off entirely: the room is darker in some areas but you have made a clear decision about what is worth keeping.

With that in mind, the two penalties can be compared directly:

| | Ridge (L2) | Lasso (L1) |
|---|---|---|
| **Penalty** | $\lambda \sum_j w_j^2$ | $\lambda \sum_j \|w_j\|$ |
| **Effect on coefficients** | Shrinks all toward zero, none reach exactly zero | Drives some to exactly zero |
| **Solution type** | Dense: all features retained | Sparse: automatic feature selection |
| **Best suited to** | Most features contribute useful signal | Few features are truly informative |

When you are unsure which to use, **Elastic Net** combines both penalties and is often a safer default: it retains Lasso's sparsity-inducing property while using the L2 term to handle correlated features more gracefully than pure Lasso alone.

---

### 14.3 Applying Regularisation in scikit-learn: A Step-by-Step Workflow

Ridge regression is one example of regularisation, but the same idea applies across many model types in scikit-learn. Decision trees, for instance, do not have a penalty term on their coefficients, but they have equivalent complexity controls: `max_depth` limits how deep the tree can grow, `min_samples_leaf` requires a minimum number of training samples in every leaf node, and `min_samples_split` controls how many samples must be present before a node is allowed to split further. All of these act as regularisation in the sense that they constrain the model's capacity and push it toward simpler solutions.

This section walks through a complete regularisation workflow using a decision tree on a synthetic dataset, showing each step explicitly. The goal is not to find the best possible model, but to demonstrate the process of fitting at multiple complexity levels, observing the bias-variance symptoms in the results, and selecting the complexity that generalises best.

**Step 1: create the dataset.** We generate a synthetic binary classification problem with a moderate degree of class overlap, enough to make the task non-trivial without making it impossible. The same `make_classification` call used earlier in this notebook is a natural starting point.

In [ ]:
# Listing 27.
# A moderately difficult binary classification problem: 10 features
# (6 informative), moderate overlap via flip_y, and a 1:3 class imbalance
# to make the metric choice matter.
N_SAMPLES_REG  = 1_500
POSITIVE_FRAC  = 0.25
RANDOM_STATE   = 0

X_reg, y_reg = make_classification(
    n_samples=N_SAMPLES_REG,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    weights=[1 - POSITIVE_FRAC, POSITIVE_FRAC],
    flip_y=0.06,
    random_state=RANDOM_STATE,
)

print(f'Dataset: {len(y_reg):,} samples  '
      f'({y_reg.sum():,} positive, {(y_reg == 0).sum():,} negative)')

**Step 2: split into training and test sets, then standardise.** The split is stratified to preserve the class ratio in both sets. Standardisation is applied here for consistency with the rest of the notebook, though decision trees are not sensitive to feature scale in the same way that logistic regression or Ridge are.

In [ ]:
# Listing 28.
X_tr_reg, X_te_reg, y_tr_reg, y_te_reg = train_test_split(
    X_reg, y_reg,
    test_size=0.30,
    stratify=y_reg,
    random_state=RANDOM_STATE,
)

scaler_reg  = StandardScaler()
X_tr_reg_s  = scaler_reg.fit_transform(X_tr_reg)
X_te_reg_s  = scaler_reg.transform(X_te_reg)

print(f'Training set : {len(y_tr_reg):,} samples')
print(f'Test set     : {len(y_te_reg):,} samples')

**Step 3: fit at multiple complexity levels.** Rather than choosing a single `max_depth` upfront, we sweep across a range of values and record both training and test performance at each level. This produces the same kind of bias-variance curve seen in Figure 10, but now evaluated using metrics appropriate for the imbalanced dataset: balanced accuracy and MCC alongside standard accuracy, so we can see whether the complexity choice affects class-imbalanced metrics differently from overall accuracy.

In [ ]:
# Listing 29.
# Sweep max_depth from 1 (maximum simplicity) to 20 (risk of overfitting).
# At each depth, fit a fresh tree and record four metrics on both sets.
depths_reg   = list(range(1, 21))
results_reg  = []

for depth in depths_reg:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE)
    dt.fit(X_tr_reg_s, y_tr_reg)

    y_pred_tr = dt.predict(X_tr_reg_s)
    y_pred_te = dt.predict(X_te_reg_s)

    results_reg.append({
        'max_depth':       depth,
        'train_accuracy':  accuracy_score(y_tr_reg, y_pred_tr),
        'test_accuracy':   accuracy_score(y_te_reg, y_pred_te),
        'train_bal_acc':   balanced_accuracy_score(y_tr_reg, y_pred_tr),
        'test_bal_acc':    balanced_accuracy_score(y_te_reg, y_pred_te),
        'train_f1':        f1_score(y_tr_reg, y_pred_tr, zero_division=0),
        'test_f1':         f1_score(y_te_reg, y_pred_te, zero_division=0),
        'train_mcc':       matthews_corrcoef(y_tr_reg, y_pred_tr),
        'test_mcc':        matthews_corrcoef(y_te_reg, y_pred_te),
    })

df_reg = pd.DataFrame(results_reg).set_index('max_depth')

# Identify the depth that maximises test MCC, the most informative single
# metric for an imbalanced binary problem.
best_depth_reg = df_reg['test_mcc'].idxmax()
print(f'Optimal max_depth by test MCC: {best_depth_reg}')
print()
print(df_reg.round(3).to_string())

**Step 4: inspect the results and select the best complexity.** The table above gives the full picture. Read each metric pair (train vs test) down the columns as `max_depth` increases: training performance should rise monotonically, while test performance peaks somewhere in the middle and then degrades. The optimal depth is the one that maximises test MCC, though you should also check that balanced accuracy and F1 tell a consistent story before committing to that choice.

In [ ]:
# Listing 30.
print(f'At max_depth = {best_depth_reg}:')
print(f'  Train accuracy    : {df_reg.loc[best_depth_reg, "train_accuracy"]:.3f}')
print(f'  Test  accuracy    : {df_reg.loc[best_depth_reg, "test_accuracy"]:.3f}')
print(f'  Train balanced acc: {df_reg.loc[best_depth_reg, "train_bal_acc"]:.3f}')
print(f'  Test  balanced acc: {df_reg.loc[best_depth_reg, "test_bal_acc"]:.3f}')
print(f'  Train F1          : {df_reg.loc[best_depth_reg, "train_f1"]:.3f}')
print(f'  Test  F1          : {df_reg.loc[best_depth_reg, "test_f1"]:.3f}')
print(f'  Train MCC         : {df_reg.loc[best_depth_reg, "train_mcc"]:.3f}')
print(f'  Test  MCC         : {df_reg.loc[best_depth_reg, "test_mcc"]:.3f}')

**Step 5: retrain the final model at the chosen complexity.** Once the optimal depth is identified from the sweep, retrain a single tree at that depth on the full training set and evaluate it once on the test set. This is the model you would take forward.

> ⚠️ **Warning:** in a real project you would not use the test set to select the optimal depth, as doing so leaks information about the test set into the model selection decision and gives an overly optimistic estimate of generalisation performance. The correct approach is to use cross-validation on the training set alone to select `max_depth`, then evaluate the final model on the test set exactly once. Cross-validation is covered in Notebook 10.

In [ ]:
# Listing 31.
# Retrain at the chosen depth on the full training set.
clf_final = DecisionTreeClassifier(max_depth=best_depth_reg, random_state=RANDOM_STATE)
clf_final.fit(X_tr_reg_s, y_tr_reg)
y_pred_final = clf_final.predict(X_te_reg_s)

print(f'Final model: DecisionTreeClassifier(max_depth={best_depth_reg})')
print(f'  Test accuracy     : {accuracy_score(y_te_reg, y_pred_final):.3f}')
print(f'  Test balanced acc : {balanced_accuracy_score(y_te_reg, y_pred_final):.3f}')
print(f'  Test F1           : {f1_score(y_te_reg, y_pred_final, zero_division=0):.3f}')
print(f'  Test MCC          : {matthews_corrcoef(y_te_reg, y_pred_final):.3f}')

---

### 14.4 Regularisation Across Different Classifiers

🔙 [Back to Table of Contents](#table-of-contents)

The Ridge and Lasso examples above both used linear models, where regularisation is expressed as an explicit penalty term added to the loss function. The decision tree example in Section 14.3 used a different mechanism entirely: constraining the tree's depth rather than penalising its coefficients. These are both forms of regularisation in the broader sense, controlling model complexity to improve generalisation, but the specific mechanism and the parameters you tune depend heavily on the type of classifier you are using.

This is a source of genuine confusion: after learning about Ridge and Lasso, it is natural to ask "where do I put the L1 or L2 penalty in my decision tree?". The answer is that you do not, because decision trees do not have coefficients to penalise. Each classifier family has its own way of expressing the bias-variance trade-off, and understanding which knobs to turn for each one is an important skill in its own right.

The table below summarises how regularisation works for common classifiers, what parameters control it, and whether L1/L2 penalties are relevant.

| Classifier | Regularisation mechanism | Key parameters | L1/L2 relevant? |
|---|---|---|---|
| Logistic Regression | Explicit penalty on coefficients | `C` (= $1/\lambda$): smaller = stronger regularisation; `penalty`: `'l1'`, `'l2'`, or `'elasticnet'` | Yes, directly |
| Linear SVM | Penalty on the margin slack variables | `C`: smaller = wider margin = stronger regularisation | Yes, directly |
| Naive Bayes | `var_smoothing` adds a small constant to feature variances, preventing zero-probability estimates | `var_smoothing` | No |
| Decision Tree | Structural constraints that limit tree growth | `max_depth`, `min_samples_leaf`, `min_samples_split`, `max_features` | No |
| Random Forest | Same structural constraints as a single tree, plus randomness across many trees reduces variance by averaging | `max_depth`, `min_samples_leaf`, `n_estimators` (more trees = lower variance) | No |
| SVM with RBF kernel | `C` controls the margin trade-off; `gamma` controls how far the influence of a single training point reaches (high gamma = high variance) | `C`, `gamma` | Indirectly via `C` |
| MLP (Neural Network) | L1/L2 weight decay on the network weights; dropout (not in sklearn); early stopping | `alpha` (L2 weight decay in sklearn's `MLPClassifier`) | Yes, via `alpha` |

### 14.4.1 Classifiers with explicit L1/L2 regularisation

For **logistic regression** and **linear SVMs**, the connection to Ridge and Lasso is direct. Logistic regression minimises a cross-entropy loss and can add either an L1 or L2 penalty to the coefficients, controlled via the `penalty` argument. The strength is set via `C = 1/λ`: scikit-learn uses `C` rather than `λ` so that larger values mean less regularisation, the opposite direction from `λ`. The default is `C=1.0` with L2, which is a moderate amount of regularisation and a sensible starting point.

```python
from sklearn.linear_model import LogisticRegression

# L2 regularisation (Ridge): shrinks all coefficients, keeps all features.
clf_l2 = LogisticRegression(penalty='l2', C=1.0, max_iter=1000)

# L1 regularisation (Lasso): drives some coefficients to zero, automatic
# feature selection. Requires solver='liblinear' or solver='saga'.
clf_l1 = LogisticRegression(penalty='l1', C=1.0, solver='liblinear', max_iter=1000)

# Elastic Net: combines L1 and L2. Requires solver='saga' and
# l1_ratio sets the mix (0 = pure L2, 1 = pure L1).
clf_en = LogisticRegression(penalty='elasticnet', C=1.0,
                            solver='saga', l1_ratio=0.5, max_iter=1000)
```

For **MLPs**, scikit-learn's `MLPClassifier` applies L2 weight decay via the `alpha` parameter (note: sklearn reuses `alpha` here for what is conceptually the same as `λ` in Ridge).

```python
from sklearn.neural_network import MLPClassifier

# alpha controls L2 weight decay on the network weights.
# Larger alpha = stronger regularisation = simpler network behaviour.
clf_mlp = MLPClassifier(hidden_layer_sizes=(100,), alpha=0.01, max_iter=500)
```

### 14.4.2 Classifiers with structural regularisation

For **decision trees**, **random forests**, and similar tree-based methods, there are no coefficients to penalise. Instead, regularisation works by constraining how the tree is allowed to grow. The most important parameters are:

```python
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Decision tree: limit depth and require a minimum number of samples
# at each leaf. Both prevent the tree from carving the data too finely.
clf_dt = DecisionTreeClassifier(
    max_depth=5,           # maximum number of splits from root to leaf
    min_samples_leaf=5,    # each leaf must contain at least 5 training samples
    min_samples_split=10,  # a node must have at least 10 samples to split
)

# Random forest: the same structural parameters apply to each tree,
# plus n_estimators controls how many trees are averaged. More trees
# reduces variance but increases training time; it does not increase bias.
clf_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=3,
)
```

### 14.4.3 Naive Bayes: a special case

Naive Bayes does not have coefficients in the same sense as a linear model, and it does not grow a structure in the way a tree does. Its main vulnerability is not overfitting in the classical sense but rather **zero-probability estimates**: if a feature value never appears in the training data for a given class, the model assigns that class a probability of zero for any instance with that feature value, regardless of all other evidence. The `var_smoothing` parameter in `GaussianNB` addresses this by adding a small constant to the estimated feature variances, preventing the model from becoming overconfident about the absence of values it has not seen.

```python
from sklearn.naive_bayes import GaussianNB

# var_smoothing adds a fraction of the largest feature variance to all
# estimated variances, preventing zero-probability estimates.
# The default of 1e-9 is usually sufficient.
clf_nb = GaussianNB(var_smoothing=1e-9)
```

### 14.4.4 The common principle

Despite the different mechanisms, all of these approaches share the same underlying logic: they introduce a constraint or cost that discourages the model from using its full capacity, nudging it away from memorising the training data and toward learning patterns that will generalise. Which regularisation mechanism is available is determined by the model family, not by your own preference, but the right strength within that mechanism is always determined by validation performance rather than by intuition alone.

> 📌 **What to notice:** the parameter names are not consistent across sklearn estimators. `C` in logistic regression and SVMs is the inverse of regularisation strength. `alpha` in Ridge, Lasso, and `MLPClassifier` is the regularisation strength directly. `max_depth` in trees is a structural constraint rather than a penalty. Reading the documentation for each estimator carefully, particularly the direction of the regularisation parameter, prevents a common class of mistakes where you think you are adding regularisation but are actually removing it.


---

## 15. Summary

### Evaluation Metrics

| Metric | Formula | When to prioritise |
|---|---|---|
| Recall (TPR / Sensitivity) | $TP / (TP + FN)$ | Missing a positive is costly (medical screening, fraud detection) |
| Precision | $TP / (TP + FP)$ | False alarms are costly (spam filtering, drug approval) |
| F1-Score | $2 \cdot P \cdot R / (P + R)$ | Both FP and FN matter; imbalanced datasets |
| Specificity (TNR) | $TN / (TN + FP)$ | Performance on the negative class |
| False Positive Rate | $FP / (FP + TN)$ | False alarm rate; x-axis of the ROC curve |
| G-Mean | $\sqrt{TPR \cdot TNR}$ | Balancing both classes on imbalanced data |
| Balanced Accuracy | $(\text{Sensitivity} + \text{Specificity}) / 2$ | Equal weighting across all classes |
| Cohen's Kappa | $(p_o - p_e) / (1 - p_e)$ | Agreement relative to random chance |
| MCC | $\frac{TP \cdot TN - FP \cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$ | Most complete single-number binary metric |

### Multi-Class Averaging

| Strategy | How | Use when |
|---|---|---|
| Macro | Simple average across classes | All classes equally important |
| Micro | Pool TP, FP, FN then compute | Larger classes should contribute more |
| Weighted | Weighted by class support | Overall performance with imbalance context |

### Bias-Variance Trade-off

| Issue | Symptom | Fix |
|---|---|---|
| **Underfitting** (high bias) | Poor training AND test accuracy | More complex model; more features; less regularisation |
| **Overfitting** (high variance) | Great training accuracy, poor test accuracy | Simpler model; more data; regularisation; cross-validation |
| **Sweet spot** | Small generalisation gap | Correct model complexity + appropriate regularisation |

### Regularisation

| Method | Penalty | Sparsity | Best for |
|---|---|---|---|
| None | — | — | Rarely appropriate |
| Ridge (L2) | $\lambda \sum w_j^2$ | Dense (shrinks all) | Most general default |
| Lasso (L1) | $\lambda \sum \|w_j\|$ | Sparse (zeros some) | Few features are truly informative |


---

## 16. References

Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer.

Chicco, D., & Jurman, G. (2020). The advantages of the Matthews correlation coefficient (MCC) over F1 score and accuracy in binary classification evaluation. *BMC Genomics*, 21(1), 6. https://doi.org/10.1186/s12864-019-6413-7

Géron, A. (2022). *Hands-on Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3rd ed.). O'Reilly Media.

Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning: Data Mining, Inference, and Prediction* (2nd ed.). Springer. https://doi.org/10.1007/978-0-387-84858-7

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M., & Duchesnay, E. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, 12, 2825-2830.

Saito, T., & Rehmsmeier, M. (2015). The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. *PLOS ONE*, 10(3), e0118432. https://doi.org/10.1371/journal.pone.0118432